In [1]:
# ============================================================
# Cell 1 — Imports and paths
# ============================================================

from pathlib import Path

import rasterio
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "gee_exports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Raw: c:\Projects\floodske\floodske\data\raw\gee_exports
Processed: c:\Projects\floodske\floodske\data\processed


In [2]:
# ============================================================
# Cell 2 — Define input raster paths
# ============================================================

raster_paths = {
    "dem": RAW_DIR / "kenya_dem_srtm_90m.tif",
    "slope": RAW_DIR / "kenya_slope_srtm_90m.tif",
    "rainfall": RAW_DIR / "kenya_chirps_rainfall_2000_2025_90m.tif",
    "landcover": RAW_DIR / "kenya_esa_worldcover_2021_90m.tif",
    "distance_to_water": RAW_DIR / "kenya_distance_to_water_90m.tif",
}

for name, path in raster_paths.items():
    print(name, "exists:", path.exists(), "|", path)

dem exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_dem_srtm_90m.tif
slope exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_slope_srtm_90m.tif
rainfall exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_chirps_rainfall_2000_2025_90m.tif
landcover exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_esa_worldcover_2021_90m.tif
distance_to_water exists: True | c:\Projects\floodske\floodske\data\raw\gee_exports\kenya_distance_to_water_90m.tif


In [3]:
# ============================================================
# Cell 3 — Validate raster metadata
# ============================================================

for name, path in raster_paths.items():
    print("\n" + "=" * 60)
    print(name.upper())

    with rasterio.open(path) as src:
        print("CRS:", src.crs)
        print("Width x Height:", src.width, "x", src.height)
        print("Bounds:", src.bounds)
        print("Resolution:", src.res)
        print("Band count:", src.count)
        print("NoData:", src.nodata)
        print("Dtype:", src.dtypes[0])


DEM
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: int16

SLOPE
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: float32

RAINFALL
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution: (0.0008084837557075694, 0.0008084837557075694)
Band count: 1
NoData: None
Dtype: float64

LANDCOVER
CRS: EPSG:4326
Width x Height: 9890 x 12482
Bounds: BoundingBox(left=33.910234165642585, bottom=-4.677078526768288, right=41.90613850959045, top=5.414415711973592)
Resolution

In [4]:
# ============================================================
# Cell 4 — Inspect raster value ranges
# ============================================================

for name, path in raster_paths.items():
    print("\n" + "=" * 60)
    print(name.upper())

    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        print("Min:", np.nanmin(arr))
        print("Max:", np.nanmax(arr))
        print("Mean:", np.nanmean(arr))
        print("NaN cells:", np.isnan(arr).sum())


DEM
Min: -12.0
Max: 5030.0
Mean: 467.91354
NaN cells: 0

SLOPE
Min: 0.0
Max: 77.88144
Mean: 3.1766622
NaN cells: 49873490

RAINFALL
Min: 512.94556
Max: 63506.438
Mean: 16474.715
NaN cells: 49873962

LANDCOVER
Min: 0.0
Max: 95.0
Mean: 15.701256
NaN cells: 0

DISTANCE_TO_WATER
Min: 0.0
Max: 89646.375
Mean: 21582.887
NaN cells: 49830317


In [5]:
# ============================================================
# Cell 5 — Check valid pixel coverage
# ============================================================

for name, path in raster_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        total_pixels = arr.size
        valid_pixels = np.count_nonzero(~np.isnan(arr))
        pct_valid = (valid_pixels / total_pixels) * 100

        print(
            f"{name:20s} "
            f"valid={valid_pixels:,} "
            f"total={total_pixels:,} "
            f"({pct_valid:.2f}%)"
        )

dem                  valid=123,446,980 total=123,446,980 (100.00%)
slope                valid=73,573,490 total=123,446,980 (59.60%)
rainfall             valid=73,573,018 total=123,446,980 (59.60%)
landcover            valid=123,446,980 total=123,446,980 (100.00%)
distance_to_water    valid=73,616,663 total=123,446,980 (59.63%)


In [6]:
# ============================================================
# Cell 6 — Compare valid coverage masks
# ============================================================

with rasterio.open(raster_paths["slope"]) as src:
    slope_mask = ~np.isnan(src.read(1).astype("float32"))

with rasterio.open(raster_paths["rainfall"]) as src:
    rainfall_mask = ~np.isnan(src.read(1).astype("float32"))

with rasterio.open(raster_paths["distance_to_water"]) as src:
    distance_mask = ~np.isnan(src.read(1).astype("float32"))

print("Slope vs Rainfall:",
      np.array_equal(slope_mask, rainfall_mask))

print("Slope vs Distance:",
      np.array_equal(slope_mask, distance_mask))

print("Rainfall vs Distance:",
      np.array_equal(rainfall_mask, distance_mask))

Slope vs Rainfall: False
Slope vs Distance: False
Rainfall vs Distance: False


In [7]:
# ============================================================
# Cell 7 — Calculate common valid modelling mask
# ============================================================

valid_masks = {}

for name, path in raster_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        valid_masks[name] = ~np.isnan(arr)

common_valid_mask = np.logical_and.reduce(list(valid_masks.values()))

total_pixels = common_valid_mask.size
common_valid_pixels = np.count_nonzero(common_valid_mask)
common_valid_pct = (common_valid_pixels / total_pixels) * 100

print("Common valid pixels:", f"{common_valid_pixels:,}")
print("Total pixels:", f"{total_pixels:,}")
print("Common valid coverage:", f"{common_valid_pct:.2f}%")

Common valid pixels: 73,533,036
Total pixels: 123,446,980
Common valid coverage: 59.57%


In [8]:
# ============================================================
# Cell 8 — Save common valid modelling mask
# ============================================================

mask_output_path = PROCESSED_DIR / "kenya_common_valid_modelling_mask.tif"

with rasterio.open(raster_paths["dem"]) as src:
    profile = src.profile.copy()

profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="lzw"
)

with rasterio.open(mask_output_path, "w", **profile) as dst:
    dst.write(common_valid_mask.astype("uint8"), 1)

print("Saved common modelling mask:")
print(mask_output_path)

Saved common modelling mask:
c:\Projects\floodske\floodske\data\processed\kenya_common_valid_modelling_mask.tif


In [10]:
# ============================================================
# Cell 9 — Define normalization helpers
# ============================================================

def read_factor(name):
    path = raster_paths[name]

    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

    arr[~common_valid_mask] = np.nan

    return arr, profile


def normalize_high_values_high_risk(arr):
    valid = arr[~np.isnan(arr)]
    min_value = np.nanmin(valid)
    max_value = np.nanmax(valid)

    return (arr - min_value) / (max_value - min_value)


def normalize_low_values_high_risk(arr):
    return 1 - normalize_high_values_high_risk(arr)

In [11]:
# ============================================================
# Cell 10 — Normalize flood factors
# ============================================================

dem_arr, base_profile = read_factor("dem")
slope_arr, _ = read_factor("slope")
rainfall_arr, _ = read_factor("rainfall")
distance_arr, _ = read_factor("distance_to_water")

dem_risk = normalize_low_values_high_risk(dem_arr)
slope_risk = normalize_low_values_high_risk(slope_arr)
rainfall_risk = normalize_high_values_high_risk(rainfall_arr)
distance_risk = normalize_low_values_high_risk(distance_arr)

print("Normalized factors created.")
print("DEM risk:", np.nanmin(dem_risk), np.nanmax(dem_risk))
print("Slope risk:", np.nanmin(slope_risk), np.nanmax(slope_risk))
print("Rainfall risk:", np.nanmin(rainfall_risk), np.nanmax(rainfall_risk))
print("Distance risk:", np.nanmin(distance_risk), np.nanmax(distance_risk))

Normalized factors created.
DEM risk: 0.0 1.0
Slope risk: 0.0 1.0
Rainfall risk: 0.0 1.0
Distance risk: 0.0 1.0


In [12]:
# ============================================================
# Cell 11 — Create flood susceptibility index
# ============================================================

WEIGHTS = {
    "dem": 0.30,
    "slope": 0.25,
    "rainfall": 0.25,
    "distance": 0.20,
}

flood_susceptibility = (
    WEIGHTS["dem"] * dem_risk +
    WEIGHTS["slope"] * slope_risk +
    WEIGHTS["rainfall"] * rainfall_risk +
    WEIGHTS["distance"] * distance_risk
)

print("Flood susceptibility created.")
print("Min:", np.nanmin(flood_susceptibility))
print("Max:", np.nanmax(flood_susceptibility))
print("Mean:", np.nanmean(flood_susceptibility))

Flood susceptibility created.
Min: 0.37609124
Max: 0.9019413
Mean: 0.7075289


In [15]:
# ============================================================
# Cell 12 — Classify flood susceptibility using quantiles
# ============================================================

valid_values = flood_susceptibility[common_valid_mask]
valid_values = valid_values[~np.isnan(valid_values)]

q20, q40, q60, q80 = np.percentile(valid_values, [20, 40, 60, 80])

print("Quantile breaks:")
print("20%:", q20)
print("40%:", q40)
print("60%:", q60)
print("80%:", q80)

susceptibility_class = np.zeros(flood_susceptibility.shape, dtype=np.uint8)

susceptibility_class[common_valid_mask & (flood_susceptibility <= q20)] = 1
susceptibility_class[common_valid_mask & (flood_susceptibility > q20) & (flood_susceptibility <= q40)] = 2
susceptibility_class[common_valid_mask & (flood_susceptibility > q40) & (flood_susceptibility <= q60)] = 3
susceptibility_class[common_valid_mask & (flood_susceptibility > q60) & (flood_susceptibility <= q80)] = 4
susceptibility_class[common_valid_mask & (flood_susceptibility > q80)] = 5

unique, counts = np.unique(susceptibility_class, return_counts=True)

for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count:,}")

Quantile breaks:
20%: 0.6601116061210632
40%: 0.6938409805297852
60%: 0.7204650640487671
80%: 0.7521359920501709
Class 0: 49,913,944
Class 1: 14,706,622
Class 2: 14,706,626
Class 3: 14,706,620
Class 4: 14,706,568
Class 5: 14,706,600


In [ ]:
# ============================================================
# Cell 13 — Save continuous and classified susceptibility rasters
# ============================================================

continuous_output = PROCESSED_DIR / "kenya_flood_susceptibility_index.tif"
classified_output = PROCESSED_DIR / "kenya_flood_susceptibility_class.tif"

index_profile = base_profile.copy()
index_profile.update(
    dtype="float32",
    count=1,
    nodata=-9999,
    compress="lzw"
)

index_to_save = np.where(
    common_valid_mask,
    flood_susceptibility,
    -9999
).astype("float32")

with rasterio.open(continuous_output, "w", **index_profile) as dst:
    dst.write(index_to_save, 1)

class_profile = base_profile.copy()
class_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="lzw"
)

with rasterio.open(classified_output, "w", **class_profile) as dst:
    dst.write(susceptibility_class.astype("uint8"), 1)

print("Saved:")
print(continuous_output)
print(classified_output)